In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

# Define our paths
DATA_DIR = '/content/drive/MyDrive/oasis1_mri/'
LABELS_EXCEL = '/content/drive/MyDrive/oasis_cross_sectional.xlsx'

# Read the clinical data spreadsheet you uploaded
df = pd.read_excel(LABELS_EXCEL)

# Function to map the clinical dementia ratings to numeric classes
def cdr_to_label(cdr):
    if cdr == 0 or pd.isna(cdr):
        return 0   # 0 = Cognitively Normal (CN)
    elif cdr == 0.5:
        return 1   # 1 = Mild Cognitive Impairment (MCI)
    else:
        return 2   # 2 = Alzheimer's Disease (AD)

# Apply the mapping
df['label'] = df['CDR'].apply(cdr_to_label)
label_map = dict(zip(df['ID'], df['label']))

print("Class Distribution across the 416 subjects:")
print(df['label'].value_counts())

In [ ]:
import nibabel as nib
import numpy as np
import os
from pathlib import Path
from skimage.transform import resize
import matplotlib.pyplot as plt

# ── Configuration ────────────────────────────────────────────────────────────
DATA_DIR = '/content/drive/MyDrive/oasis1_mri/'

# OASIS-1 structural scan filename patterns:
#   - "anon"  → anonymized raw anatomy files  (e.g., OAS1_0001_MR1_anon.img)
#   - "mpr"   → motion-corrected average scan (e.g., OAS1_0001_MR1_mpr_n4_anon.img)
# We prefer "mpr" when available (averaged, higher SNR); fall back to "anon".
PREFERRED_PATTERN = "mpr"
FALLBACK_PATTERN  = "anon"


def extract_middle_slice(mri_path: Path) -> np.ndarray:
    """
    Load a raw MRI volume (.img/.hdr pair) and return its middle axial
    slice as a normalised, resized float32 array ready for ResNet input.

    nibabel automatically locates the paired .hdr file when given the .img path.
    """
    img  = nib.load(str(mri_path))
    data = img.get_fdata()

    # Normalize intensity values to [0, 1]; epsilon avoids division by zero
    # on edge-case constant volumes
    data_min, data_max = data.min(), data.max()
    data = (data - data_min) / (data_max - data_min + 1e-8)

    # Middle axial slice (z-axis = axis 2) captures the most diagnostic
    # brain structure for Alzheimer's classification
    mid_z     = data.shape[2] // 2
    slice_2d  = data[:, :, mid_z]

    # Resize to 224×224 — the standard input size for ResNet-family models
    slice_resized = resize(slice_2d, (224, 224), anti_aliasing=True)

    return slice_resized.astype(np.float32)


def find_best_scan(subject_dir: Path) -> Path | None:
    """
    Return the best structural scan file in a subject folder.

    Priority: mpr (motion-corrected average) > anon (raw anatomy).
    Returns None if neither pattern is found.
    """
    for pattern in (PREFERRED_PATTERN, FALLBACK_PATTERN):
        candidates = list(subject_dir.rglob(f"*{pattern}*.img"))
        if candidates:
            return candidates[0]   # Take the first match; one scan per subject
    return None


# ── Discovery ────────────────────────────────────────────────────────────────
data_root = Path(DATA_DIR)

# Each subject lives in a folder named OASIS1_XXXX_MRY
subject_dirs = sorted([
    d for d in data_root.iterdir()
    if d.is_dir() and "OAS1_" in d.name
])

print(f"Found {len(subject_dirs)} subject folders. Starting slice extraction...")

# ── Extraction ───────────────────────────────────────────────────────────────
slices  = {}      # { subject_id : np.ndarray (224, 224) }
skipped = []      # subject IDs with no usable scan file
errors  = []      # (subject_id, error_message) for failed loads

for subject_dir in subject_dirs:
    subject_id = subject_dir.name   # e.g. "OAS1_0001_MR1"

    scan_file = find_best_scan(subject_dir)

    if scan_file is None:
        skipped.append(subject_id)
        continue

    try:
        slices[subject_id] = extract_middle_slice(scan_file)
    except Exception as e:
        errors.append((subject_id, str(e)))

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n── Extraction complete ──────────────────────────────────")
print(f"  Processed : {len(slices):>4d} subjects")
print(f"  Skipped   : {len(skipped):>4d}  (no .img scan found)")
print(f"  Errors    : {len(errors):>4d}  (load failure)")

if skipped:
    print(f"\nSkipped subjects (no scan file): {skipped[:10]}"
          + (" …" if len(skipped) > 10 else ""))

if errors:
    print("\nFailed subjects:")
    for sid, msg in errors[:10]:
        print(f"  {sid}: {msg}")
    if len(errors) > 10:
        print("  …")

# ── Sanity-check visualisation ───────────────────────────────────────────────
if slices:
    n_samples = min(4, len(slices))
    sample_ids = list(slices.keys())[:n_samples]

    fig, axes = plt.subplots(1, n_samples, figsize=(5 * n_samples, 5))
    if n_samples == 1:
        axes = [axes]

    for ax, sid in zip(axes, sample_ids):
        ax.imshow(slices[sid], cmap='gray')
        ax.set_title(sid, fontsize=9)
        ax.axis('off')

    fig.suptitle("Sanity Check: Middle Axial Slices (224×224, normalised)",
                 fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()

    print(f"\nSlice shape  : {next(iter(slices.values())).shape}")
    print(f"Value range  : [{next(iter(slices.values())).min():.3f}, "
          f"{next(iter(slices.values())).max():.3f}]")
    print("\n`slices` dict is ready — pass it to AlzheimerDataset.")
else:
    print("\nNo slices extracted. Check DATA_DIR and folder structure.")

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split


class AlzheimerDataset(Dataset):
    """
    PyTorch Dataset for OASIS-1 MRI slices.

    Expects:
        slices_dict : { subject_id -> np.ndarray (224, 224) or (224, 224, 1) }
        labels_dict : { subject_id -> int  (0=CN, 1=MCI, 2=AD) }
    """

    def __init__(self, subject_ids, slices_dict, labels_dict, transform=None):
        # Only keep subjects that exist in BOTH dicts
        self.subject_ids = [
            sid for sid in subject_ids
            if sid in slices_dict and sid in labels_dict
        ]
        self.slices_dict = slices_dict
        self.labels_dict = labels_dict
        self.transform   = transform

    def __len__(self):
        return len(self.subject_ids)

    def __getitem__(self, idx):
        sid = self.subject_ids[idx]
        img = self.slices_dict[sid].copy()   # copy avoids mutating the cache

        # ── Ensure shape is (H, W, 3) for ToTensor → (3, H, W) ──────────────
        if img.ndim == 2:                    # (H, W)      → (H, W, 3)
            img = np.stack([img, img, img], axis=-1)
        elif img.ndim == 3 and img.shape[-1] == 1:   # (H, W, 1) → (H, W, 3)
            img = np.concatenate([img, img, img], axis=-1)
        # If already (H, W, 3), nothing to do

        # Clip to [0, 1] before handing to ToTensor (skimage resize can
        # produce tiny floats outside this range due to anti-aliasing)
        img = np.clip(img, 0.0, 1.0).astype(np.float32)

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)   # (H, W, 3) → (3, H, W)

        label = self.labels_dict[sid]
        return img, torch.tensor(label, dtype=torch.long)


# ── 1. Transforms ─────────────────────────────────────────────────────────────
# ImageNet mean/std — correct for pretrained ResNet weights
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToTensor(),                          # (H,W,3) → (3,H,W), scales to [0,1]
    transforms.RandomHorizontalFlip(p=0.5),         # brain is roughly symmetric L–R
    transforms.RandomRotation(degrees=10),          # small rotational variance
    transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])

# ── 2. Stratified splits (70 / 15 / 15) ───────────────────────────────────────
# Stratify on label so each split has proportional CN / MCI / AD counts.
# This matters for OASIS-1 which is heavily skewed toward CN.

all_ids     = [sid for sid in slices.keys() if sid in label_map]
all_labels  = [label_map[sid] for sid in all_ids]

train_ids, temp_ids, _, temp_labels = train_test_split(
    all_ids, all_labels,
    test_size=0.30, stratify=all_labels, random_state=42
)
val_ids, test_ids, _, _ = train_test_split(
    temp_ids, temp_labels,
    test_size=0.50, stratify=temp_labels, random_state=42
)

# Each split gets its OWN Dataset instance → no shared transform reference
train_dataset = AlzheimerDataset(train_ids, slices, label_map, transform=train_transform)
val_dataset   = AlzheimerDataset(val_ids,   slices, label_map, transform=val_transform)
test_dataset  = AlzheimerDataset(test_ids,  slices, label_map, transform=val_transform)

# ── 3. DataLoaders ────────────────────────────────────────────────────────────
BATCH_SIZE = 16   # safe default for Colab A100; increase to 32 if memory allows

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, drop_last=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

# ── 4. Summary ────────────────────────────────────────────────────────────────
print("PyTorch Datasets created successfully!")
print(f"  Training   : {len(train_dataset):>4d} scans  ({len(train_loader)} batches)")
print(f"  Validation : {len(val_dataset):>4d} scans  ({len(val_loader)} batches)")
print(f"  Test       : {len(test_dataset):>4d} scans  ({len(test_loader)} batches)")
print(f"  Batch size : {BATCH_SIZE}")

# Quick shape check — catches channel or resize issues early
sample_img, sample_label = train_dataset[0]
print(f"\nSample tensor shape : {sample_img.shape}   (expect [3, 224, 224])")
print(f"Sample label        : {sample_label.item()}  (0=CN, 1=MCI, 2=AD)")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torchvision import models
from sklearn.utils.class_weight import compute_class_weight

# ── 1. Hardware Device ────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── 2. Class Weights (computed from actual training labels) ───────────────────
# Derive counts from the real train split — never hardcode these
train_labels = [label_map[sid] for sid in train_ids if sid in label_map]

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_labels
)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

label_names = {0: "CN", 1: "MCI", 2: "AD"}
counts = {label_names[i]: train_labels.count(i) for i in range(3)}
print(f"Training label counts : {counts}")
print(f"Class weights         : CN={class_weights[0]:.3f}  MCI={class_weights[1]:.3f}  AD={class_weights[2]:.3f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── 3. ResNet-18 with Frozen Backbone ─────────────────────────────────────────
# Strategy: freeze all pretrained layers, train only the new classifier head.
# With ~300 training samples, fine-tuning the full backbone immediately causes
# overfitting. After a few epochs of head-only training you can unfreeze
# (see the unfreeze block below) for a second-stage fine-tune.

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze every layer in the pretrained backbone
for param in model.parameters():
    param.requires_grad = False

# Replace the final FC layer: 512 → Dropout → 3 classes
num_ftrs = model.fc.in_features   # 512 for ResNet-18
model.fc = nn.Sequential(
    nn.Dropout(p=0.4),            # regularization — important on small datasets
    nn.Linear(num_ftrs, 3)        # CN=0, MCI=1, AD=2
)

model = model.to(device)

# ── 4. Optimizer and Scheduler ────────────────────────────────────────────────
# Only pass the classifier head parameters — backbone is frozen
EPOCHS = 25

optimizer = torch.optim.AdamW(
    model.fc.parameters(),        # head only during frozen stage
    lr=1e-3,
    weight_decay=1e-4             # mild L2; 1e-2 is too aggressive with AdamW
)

# Cosine decay: smoothly anneals LR over training without reacting to
# val-loss noise the way ReduceLROnPlateau does on small datasets
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# ── 5. Optional: Unfreeze Backbone for Stage-2 Fine-Tuning ───────────────────
# Run this block after ~10 epochs of head-only training, then continue training
# for another 10-15 epochs with a lower LR across all parameters.
#
# for param in model.parameters():
#     param.requires_grad = True
#
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

# ── 6. Summary ────────────────────────────────────────────────────────────────
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\nModel      : ResNet-18 (backbone frozen)")
print(f"Parameters : {trainable:,} trainable / {total:,} total")
print(f"Optimizer  : AdamW  lr=1e-3  weight_decay=1e-4")
print(f"Scheduler  : CosineAnnealingLR  T_max={EPOCHS}")
print(f"Loss       : CrossEntropyLoss with class weights")
print("\nModel ready for training.")

In [ ]:
import time
import copy
from sklearn.metrics import f1_score

# ── Configuration ─────────────────────────────────────────────────────────────
CHECKPOINT_PATH = '/content/drive/MyDrive/best_alzheimer_model.pt'
EARLY_STOP_PATIENCE = 7   # stop if val loss doesn't improve for this many epochs

best_val_loss    = float('inf')
best_model_state = None
epochs_no_improve = 0

history = {
    'train_loss': [], 'train_acc': [], 'train_f1': [],
    'val_loss':   [], 'val_acc':   [], 'val_f1':   []
}

print("Starting Training — Stage 1: Head Only")
print(f"Early stopping patience: {EARLY_STOP_PATIENCE} epochs\n")

for epoch in range(EPOCHS):
    start_time = time.time()

    # ── Training Phase ─────────────────────────────────────────────────────────
    model.train()
    train_loss      = 0.0
    train_corrects  = 0
    train_samples   = 0          # track actual samples seen (drop_last=True skips last batch)
    all_train_preds, all_train_labels = [], []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size      = inputs.size(0)
        train_loss     += loss.item() * batch_size
        train_samples  += batch_size

        _, preds = torch.max(outputs, 1)
        train_corrects += (preds == labels).sum().item()

        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())

    scheduler.step()

    epoch_train_loss = train_loss / train_samples
    epoch_train_acc  = train_corrects / train_samples
    epoch_train_f1   = f1_score(
        all_train_labels, all_train_preds,
        average='macro', zero_division=0   # consistent with val
    )

    # ── Validation Phase ───────────────────────────────────────────────────────
    model.eval()
    val_loss     = 0.0
    val_corrects = 0
    val_samples  = 0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            batch_size    = inputs.size(0)
            val_loss     += loss.item() * batch_size
            val_samples  += batch_size

            _, preds = torch.max(outputs, 1)
            val_corrects += (preds == labels).sum().item()

            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / val_samples
    epoch_val_acc  = val_corrects / val_samples
    epoch_val_f1   = f1_score(
        all_val_labels, all_val_preds,
        average='macro', zero_division=0
    )

    # ── History ────────────────────────────────────────────────────────────────
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['train_f1'].append(epoch_train_f1)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(epoch_val_f1)

    # ── Checkpoint + Early Stopping ────────────────────────────────────────────
    improved = epoch_val_loss < best_val_loss
    if improved:
        best_val_loss    = epoch_val_loss
        best_model_state = copy.deepcopy(model.state_dict())   # deep copy — not a reference
        torch.save(best_model_state, CHECKPOINT_PATH)          # persist to Drive
        epochs_no_improve = 0
        flag = "  [saved]"
    else:
        epochs_no_improve += 1
        flag = f"  [no improvement {epochs_no_improve}/{EARLY_STOP_PATIENCE}]"

    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch+1:02d}/{EPOCHS}  [{elapsed:.1f}s]  "
        f"Train loss {epoch_train_loss:.3f}  acc {epoch_train_acc:.3f}  f1 {epoch_train_f1:.3f}  |  "
        f"Val loss {epoch_val_loss:.3f}  acc {epoch_val_acc:.3f}  f1 {epoch_val_f1:.3f}"
        + flag
    )

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1} — val loss hasn't improved in {EARLY_STOP_PATIENCE} epochs.")
        break

# ── Restore Best Weights ───────────────────────────────────────────────────────
model.load_state_dict(best_model_state)
print(f"\nStage 1 complete. Best val loss: {best_val_loss:.4f}. Best weights restored.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, f1_score
)
from sklearn.preprocessing import label_binarize
import seaborn as sns
import numpy as np

TARGET_NAMES   = ['CN', 'MCI', 'AD']
TARGET_NAMES_LONG = [
    'Cognitively Normal (CN)',
    'Mild Cognitive Impairment (MCI)',
    "Alzheimer's Disease (AD)"
]

# ── 1. Training Curves ────────────────────────────────────────────────────────
epochs_range = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], label='Train', color='#1f77b4', lw=2)
ax1.plot(epochs_range, history['val_loss'],   label='Val',   color='#ff7f0e', lw=2, linestyle='--')
ax1.set_title('Loss', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

ax2.plot(epochs_range, history['train_f1'], label='Train', color='#2ca02c', lw=2)
ax2.plot(epochs_range, history['val_f1'],   label='Val',   color='#d62728', lw=2, linestyle='--')
ax2.set_title('Macro F1-Score', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle('Training History', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2. Evaluate on Held-Out Test Set ─────────────────────────────────────────
model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)

        probs = torch.softmax(outputs, dim=1)          # probabilities for AUC-ROC
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)

# ── 3. Metrics Report ─────────────────────────────────────────────────────────
print("TEST SET EVALUATION")
print("=" * 62)
print(classification_report(
    all_labels, all_preds,
    target_names=TARGET_NAMES_LONG,
    zero_division=0
))

# AUC-ROC — the primary metric for medical classifiers
# Requires ≥2 classes present in test labels; guard against edge case
unique_classes = np.unique(all_labels)
if len(unique_classes) >= 2:
    auc = roc_auc_score(
        label_binarize(all_labels, classes=[0, 1, 2]),
        all_probs,
        multi_class='ovr',
        average='macro'
    )
    print(f"Macro AUC-ROC : {auc:.4f}   (random = 0.50, target ≥ 0.80)")
else:
    print("AUC-ROC: skipped — test set contains only one class (check your split)")

macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
print(f"Macro F1      : {macro_f1:.4f}")
print("=" * 62)

# ── 4. Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

# Normalised version alongside raw counts — easier to spot per-class recall
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES, ax=ax1)
ax1.set_title('Counts', fontsize=11, fontweight='bold')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES, ax=ax2)
ax2.set_title('Recall (row-normalised)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')

plt.suptitle('Test Set Confusion Matrix', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("Unfreezing backbone for Stage 2 fine-tuning...")

# ── 1. Unfreeze all parameters ────────────────────────────────────────────────
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parameters : {trainable:,} / {total:,} now trainable")

# ── 2. Differential learning rates ───────────────────────────────────────────
# Backbone layers already learned good general features — they need a much
# smaller nudge than the classifier head, which still has room to improve.
# Using a single lr=1e-5 for all layers works but is slightly suboptimal;
# differential LRs let the head adapt faster while the backbone shifts gently.
STAGE2_EPOCHS = 15

optimizer = torch.optim.AdamW([
    {'params': model.layer1.parameters(), 'lr': 1e-6},
    {'params': model.layer2.parameters(), 'lr': 1e-6},
    {'params': model.layer3.parameters(), 'lr': 1e-5},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.fc.parameters(),     'lr': 1e-4},   # head trains fastest
], weight_decay=1e-4)

# ── 3. Fresh cosine scheduler over Stage 2 epochs ────────────────────────────
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS)

# ── 4. Reset early stopping — don't carry Stage 1 state into Stage 2 ─────────
best_val_loss     = float('inf')
best_model_state  = None
epochs_no_improve = 0

# Keep history continuous so your loss/F1 plots span both stages
# (the training loop appends to the same `history` dict)

print(f"Stage 2 ready — {STAGE2_EPOCHS} epochs, differential LRs")
print(f"  layer1/2 : lr=1e-6")
print(f"  layer3/4 : lr=1e-5")
print(f"  fc head  : lr=1e-4")
print("Run the training loop cell to begin Stage 2.")

In [ ]:
import time
import copy
from sklearn.metrics import f1_score

print("Starting Stage 2 — Full Fine-Tuning (backbone unfrozen)")
print(f"Early stopping patience: {EARLY_STOP_PATIENCE} epochs\n")

for epoch in range(STAGE2_EPOCHS):
    start_time = time.time()

    # ── Training Phase ─────────────────────────────────────────────────────────
    model.train()
    train_loss, train_corrects, train_samples = 0.0, 0, 0
    all_train_preds, all_train_labels = [], []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size      = inputs.size(0)
        train_loss     += loss.item() * batch_size
        train_samples  += batch_size
        _, preds = torch.max(outputs, 1)
        train_corrects += (preds == labels).sum().item()
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())

    scheduler.step()

    epoch_train_loss = train_loss / train_samples
    epoch_train_acc  = train_corrects / train_samples
    epoch_train_f1   = f1_score(all_train_labels, all_train_preds, average='macro', zero_division=0)

    # ── Validation Phase ───────────────────────────────────────────────────────
    model.eval()
    val_loss, val_corrects, val_samples = 0.0, 0, 0
    all_val_preds, all_val_labels = [], []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            batch_size    = inputs.size(0)
            val_loss     += loss.item() * batch_size
            val_samples  += batch_size
            _, preds = torch.max(outputs, 1)
            val_corrects += (preds == labels).sum().item()
            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    epoch_val_loss = val_loss / val_samples
    epoch_val_acc  = val_corrects / val_samples
    epoch_val_f1   = f1_score(all_val_labels, all_val_preds, average='macro', zero_division=0)

    # ── History ────────────────────────────────────────────────────────────────
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['train_f1'].append(epoch_train_f1)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    history['val_f1'].append(epoch_val_f1)

    # ── Checkpoint + Early Stopping ────────────────────────────────────────────
    if epoch_val_loss < best_val_loss:
        best_val_loss    = epoch_val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, CHECKPOINT_PATH)
        epochs_no_improve = 0
        flag = "  [saved]"
    else:
        epochs_no_improve += 1
        flag = f"  [no improvement {epochs_no_improve}/{EARLY_STOP_PATIENCE}]"

    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch+1:02d}/{STAGE2_EPOCHS}  [{elapsed:.1f}s]  "
        f"Train loss {epoch_train_loss:.3f}  acc {epoch_train_acc:.3f}  f1 {epoch_train_f1:.3f}  |  "
        f"Val loss {epoch_val_loss:.3f}  acc {epoch_val_acc:.3f}  f1 {epoch_val_f1:.3f}"
        + flag
    )

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}.")
        break

# ── Restore Best Weights ───────────────────────────────────────────────────────
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nStage 2 complete. Best val loss: {best_val_loss:.4f}. Best weights restored.")
else:
    print("\nWarning: no checkpoint was saved — re-run the Stage 2 setup cell before training.")

In [ ]:
!pip install gradio -q

import gradio as gr
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import models, transforms

# ── 1. Load Model ─────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

inference_model = models.resnet18()
num_ftrs = inference_model.fc.in_features
inference_model.fc = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(num_ftrs, 3)
)

MODEL_PATH = '/content/drive/MyDrive/best_alzheimer_model.pt'
inference_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device, weights_only=True)  # safe loading
)
inference_model.to(device)
inference_model.eval()

# ── 2. Inference Transform ────────────────────────────────────────────────────
# Must exactly match val_transform used during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

CLASS_NAMES = [
    'Cognitively Normal (CN)',
    'Mild Cognitive Impairment (MCI)',
    "Alzheimer's Disease (AD)"
]
CONFIDENCE_THRESHOLD = 0.50   # flag predictions below this

# ── 3. Prediction Function ────────────────────────────────────────────────────
def predict_mri(input_image):
    if input_image is None:
        return {"Error: no image uploaded": 1.0}

    # Gradio passes a numpy array (H, W, 3) uint8 — convert cleanly via PIL
    img = Image.fromarray(input_image).convert('RGB')
    img_tensor = inference_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = inference_model(img_tensor)
        probs   = torch.softmax(outputs, dim=1)[0]

    top_prob, top_idx = probs.max(0)

    results = {CLASS_NAMES[i]: float(probs[i]) for i in range(3)}

    # Low-confidence warning — important for a medical classifier
    if top_prob.item() < CONFIDENCE_THRESHOLD:
        results[f"⚠️ Low confidence ({top_prob.item():.0%}) — result unreliable"] = 0.0

    return results

# ── 4. Gradio Interface ───────────────────────────────────────────────────────
interface = gr.Interface(
    fn=predict_mri,
    inputs=gr.Image(
        label="Upload a 2D MRI axial slice (PNG or JPG)",
        type="numpy"
    ),
    outputs=gr.Label(
        num_top_classes=3,
        label="Diagnostic Confidence"
    ),
    title="Alzheimer's Disease MRI Classifier",
    description=(
        "Upload a pre-extracted 2D axial MRI slice from the OASIS-1 dataset. "
        "The fine-tuned ResNet-18 model returns probability estimates for three diagnostic categories: "
        "Cognitively Normal (CN), Mild Cognitive Impairment (MCI), and Alzheimer's Disease (AD).\n\n"
        "**Note:** This is a research prototype trained on ~300 samples. "
        "It is not a clinical diagnostic tool."
    ),
    examples=None,    # add paths to sample slices here once you have PNGs saved
    theme="soft",
    flagging_mode="never"
)

# share=True generates a public gradio.live URL valid for 72 hours
# debug=False keeps your tracebacks private on the public link
interface.launch(share=True, debug=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load('/content/oasis1_slices.npz')
imgs, labels, ids = d['imgs'], d['labels'], d['ids']

print(imgs.shape, imgs.dtype, imgs.min(), imgs.max())
print({0: 'CN', 1: 'MCI', 2: 'AD'})
print(np.bincount(labels))

fig, ax = plt.subplots(3, 6, figsize=(14, 7))
for row, cls in enumerate([0, 1, 2]):
    idx = np.where(labels == cls)[0][:6]
    for col, i in enumerate(idx):
        ax[row, col].imshow(imgs[i], cmap='gray')
        ax[row, col].set_title(f'{ids[i]}  {["CN","MCI","AD"][cls]}', fontsize=8)
        ax[row, col].axis('off')
plt.tight_layout()